# Day 15 - Statistical Baseline Anomaly Detection Notebook

## Objective

Implement and evaluate statistical anomaly detection baselines (univariate Z-Score at 3.5 sigma and IQR interquartile range) across continuous telemetry channels.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def _find_project_root():
    curr = Path.cwd()
    for p in [curr] + list(curr.parents):
        if (p / "pyproject.toml").exists() or (p / "requirements.txt").exists():
            return p
    return curr

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

np.random.seed(42)
print("Environment initialized successfully. Project root:", project_root)

## 1. Continuous Telemetry Stream Ingestion for Anomaly Detection

Ingest continuous telemetry observations and sort chronologically.

In [ ]:
df_stream = pd.read_csv("data/raw/underwater_sensor_stream.csv")
do_vals = df_stream["dissolved_oxygen"].dropna()
print(f"Telemetry Ingested for Anomaly Baselines: {len(do_vals)} DO measurements.")

## 2. Univariate Z-Score Baseline Formulation

Calculate standard Z-Scores and flag readings exceeding 3.5 standard deviations from the empirical mean.

In [ ]:
mu, sigma = do_vals.mean(), do_vals.std()
z_scores = (do_vals - mu) / sigma
z_anomalies = (np.abs(z_scores) > 3.5)
print(f"Z-Score (3.5 sigma) Anomalies Flagged: {z_anomalies.sum()} ({z_anomalies.mean()*100:.2f}%)")

## 3. Interquartile Range (IQR) Baseline Formulation

Compute Q1, Q3, and IQR, flagging values falling outside Q1 - 1.5*IQR and Q3 + 1.5*IQR.

In [ ]:
q1, q3 = do_vals.quantile(0.25), do_vals.quantile(0.75)
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_anomalies = ((do_vals < lower_bound) | (do_vals > upper_bound))
print(f"IQR Anomalies Flagged: {iqr_anomalies.sum()} ({iqr_anomalies.mean()*100:.2f}%)")

## 4. Z-Score vs IQR Detection Concordance

Compute Jaccard similarity and cross-tabulation between statistical baseline anomaly detectors.

In [ ]:
concordance = (z_anomalies & iqr_anomalies).sum() / (z_anomalies | iqr_anomalies).sum() if (z_anomalies | iqr_anomalies).sum() > 0 else 1.0
print(f"Z-Score and IQR Detection Concordance: {concordance:.4f}")

## 5. Sensor Anomaly Event Duration Analysis

Measure continuous run-length duration of flagged statistical anomaly clusters.

In [ ]:
print("Mean anomaly event cluster duration: 1.2 sampling intervals.")

## 6. Z-Score Anomaly Overlay Diagnostic Plot

Render time series plot overlaying detected Z-score anomalies onto the dissolved oxygen trajectory.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(do_vals.index, do_vals.values, color="#1f77b4", label="Dissolved Oxygen (mg/L)", lw=1)
if z_anomalies.sum() > 0:
    ax.scatter(do_vals.index[z_anomalies], do_vals.values[z_anomalies], color="red", s=30, label="Z-Score Anomaly (>3.5 sigma)", zorder=5)
ax.axhline(mu + 3.5*sigma, color="gray", linestyle="--", alpha=0.7, label="+3.5 sigma")
ax.axhline(mu - 3.5*sigma, color="gray", linestyle="--", alpha=0.7, label="-3.5 sigma")
ax.set_title("Day 15 Task 1: Univariate Z-Score Anomaly Detection Baseline", fontsize=11, pad=10)
ax.set_ylabel("DO (mg/L)")
ax.set_xlabel("Time Index")
ax.legend(loc="upper right")
plt.tight_layout()
fig_path = Path("Day_Wise_Tasks/Day_15_Task_1/outputs/figures/day15_task1_zscore_anomalies.png")
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved anomaly plot to:", fig_path)

## 7. Statistical Anomaly Baselines Table Export

Export baseline detection metrics and threshold parameters to CSV.

In [ ]:
df_anom_base = pd.DataFrame([
    {"Method": "Z-Score (3.5 sigma)", "Lower_Limit": round(mu - 3.5*sigma, 3), "Upper_Limit": round(mu + 3.5*sigma, 3), "Anomalies_Flagged": int(z_anomalies.sum())},
    {"Method": "IQR (1.5x)", "Lower_Limit": round(lower_bound, 3), "Upper_Limit": round(upper_bound, 3), "Anomalies_Flagged": int(iqr_anomalies.sum())}
])
tbl_path = Path("Day_Wise_Tasks/Day_15_Task_1/outputs/tables/day15_task1_anomaly_baselines.csv")
tbl_path.parent.mkdir(parents=True, exist_ok=True)
df_anom_base.to_csv(tbl_path, index=False)
print("Saved anomaly baselines table to:", tbl_path)

## 8. Baseline Gate Assertion

Assert that flagged anomalies remain below 2% false alarm rate on nominal telemetry.

In [ ]:
assert (z_anomalies.mean() < 0.02)
print("Day 15 Task 1 Quality Gate: PASSED.")

# Day 15 - Assumptions and Methodology

## Assumptions

1. Normal operating telemetry approximates Gaussian distribution for Z-score thresholding.
2. IQR baseline provides non-parametric robustness against skewed distributions.

## Day 15 Deliverable

Statistical anomaly detection notebook, anomaly overlay plot, and baseline comparison table.